<a href="https://colab.research.google.com/github/almagomen/v2v/blob/main/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video2Video – Despliegue en Colab

1. **Activa GPU**: *Runtime → Change runtime type → T4 GPU*
2. Ejecuta las celdas en orden.
3. Configura **Secrets** (panel izquierdo 🔑): `GOOGLE_API_KEY`, `LMNT_API_KEY`

## 1. Clonar repo e instalar dependencias

In [ ]:
# @title ⚠️ Clonar, Compilar y Lanzar App

import os, time, threading, subprocess, sys, warnings
from google.colab import output, userdata, drive
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)

# --- CONFIGURACIÓN ---
REPO_PATH = "/content/v2v"
BRANCH_NAME = "openreel-cursor"
GHP_TOKEN = userdata.get('GHP_GITHUB_TOKEN')
REPO_URL = f"https://{GHP_TOKEN}@github.com/almagomen/v2v.git"
FRONTEND_DIR = os.path.join(REPO_PATH, "frontend")
BACKEND_PORT = 7860
FRONTEND_PORT = 5173

# Montaje de Drive
if not os.path.exists('/content/drive'): drive.mount("/content/drive")

# Gestión del Repositorio
if not os.path.exists(REPO_PATH):
    print("📥 Clonando repositorio...")
    !git clone -q -b {BRANCH_NAME} {REPO_URL} {REPO_PATH}
else:
    print("🔄 Actualizando repositorio...")
    !git -C {REPO_PATH} reset --hard
    !git -C {REPO_PATH} fetch origin {BRANCH_NAME} -q
    !git -C {REPO_PATH} checkout {BRANCH_NAME} -q
    !git -C {REPO_PATH} pull origin {BRANCH_NAME}

if REPO_PATH not in sys.path: sys.path.append(REPO_PATH)

# Instalación de Dependencias
if 'deps_installed' not in globals():
    print("📦 Instalando dependencias...")
    !uv pip install git+https://github.com/jianfch/stable-ts.git git+https://github.com/pyannote/pyannote-audio.git --no-deps -q
    !uv pip install "yt-dlp[default]" bs-roformer-infer tinydb sentencex torchcodec==0.7.0  torch-audiomentations>=0.12.0  torchmetrics>=1.6.1 optuna>=4.2.0 -q
    !uv pip install openai-whisper 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q
    !uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0'  'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1'  -q

    print("🛠️ Cargando modelos...")
    from src.stt import get_whisper_model
    from src.diarization import get_diarization_model
    from src.download import get_roformer_model
    get_whisper_model()
    get_diarization_model()
    get_roformer_model()
    globals()['deps_installed'] = True


!npm install -g pnpm
%cd /content/v2v
!git clone https://github.com/almagomen/openreel-video.git
%cd /content/v2v/openreel-video
!pnpm install
!pnpm build
%cd ..
# !npm install --prefix {FRONTEND_DIR} --silent

# --- CLASE PARA CONTROLAR EL SERVIDOR ---
import uvicorn
class ControlledServer(uvicorn.Server):
    def install_setup(self): pass

def run_backend_logic():
    global server_instance
    from api.main import app
    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT)
    server_instance = ControlledServer(config=config)
    server_instance.run()

# Apagado si ya existía una instancia previa
if 'server_instance' in globals() and server_instance.started:
    server_instance.should_exit = True
    time.sleep(2)

# Lanzar BACKEND
back_thread = threading.Thread(target=run_backend_logic, daemon=True)
back_thread.start()

# Lanzar FRONTEND
# print("\n🛠️ Iniciando APP...")
# !fuser -k {FRONTEND_PORT}/tcp 2>/dev/null
# process = subprocess.Popen(
#     ["npm", "run", "dev", "--prefix", FRONTEND_DIR, "--", "--host"],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
# )

# if 'vite_history' not in globals(): vite_history = []
# def log_reader(pipe):
#     for line in iter(pipe.readline, ''):
#         clean_line = line.strip()
#         if clean_line: vite_history.append(clean_line)

# threading.Thread(target=log_reader, args=(process.stdout,), daemon=True).start()

print("\n✨ ENTORNO LISTO ✨")
output.serve_kernel_port_as_window(FRONTEND_PORT, path="/", skip_warning=True)